In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset,DataLoader


In [ ]:
torch.manual_seed(42)

In [ ]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [ ]:
from torchvision import transforms
transform=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean = (0.4914, 0.4822, 0.4465), std  = (0.2023, 0.1994, 0.2010))
])

In [ ]:
from torchvision import datasets
train_data=datasets.CIFAR10(root='./data',
                            train=True,
                            download=True,
                            transform=transform)

test_data=datasets.CIFAR10(root='./data',
                            train=False,
                            download=True,
                            transform=transform)

Train_Dataloader=DataLoader(train_data,batch_size=32,shuffle=True)
Test_Dataloader=DataLoader(test_data,batch_size=32,shuffle=False)


In [ ]:
image, label = train_data[0]

print(image.shape)   # (1, 28, 28)
print(label)

torch.Size([3, 32, 32])
6


In [ ]:
class CIFAR_Model(nn.Module):

  def __init__(self,in_channel):
    super().__init__()

    self.features=nn.Sequential(
        nn.Conv2d(in_channel,out_channels=32,kernel_size=3,padding=1),
        nn.BatchNorm2d(32),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2,stride=2),                  # (32-2)/2 +1=16      so the output size will be 16

        nn.Conv2d(32,64,kernel_size=3,padding=1),
        nn.BatchNorm2d(64),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2,stride=2),

        nn.Conv2d(64,128,kernel_size=3,padding=1),
        nn.BatchNorm2d(128),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2,stride=2),
    )

    self.classifier=nn.Sequential(
        nn.Flatten(),
        nn.Linear(128*4*4,128),
        nn.BatchNorm1d(128),
        nn.ReLU(),
        nn.Dropout(0.3),

        nn.Linear(128,64),
        nn.BatchNorm1d(64),
        nn.ReLU(),
        nn.Dropout(0.3),

        nn.Linear(64,10)
    )

  def forward(self,data):
    data=self.features(data)
    data=self.classifier(data)
    return data

In [ ]:
# Parameters
learning_rate=0.07
epochs=10

# model object
model=CIFAR_Model(3)
model.to(device)

# loss
criterion=nn.CrossEntropyLoss()

# optimizer
optimizer=optim.Ad(model.parameters(),lr=learning_rate)

In [ ]:
# training loop
model.train()

for epoch in range(epochs):
  total_loss=0
  for feature,label in Train_Dataloader:

    feature,label=feature.to(device),label.to(device)

    # forward pass
    y_pred=model(feature)

    # loss
    loss=criterion(y_pred,label)

    # backward
    optimizer.zero_grad()
    loss.backward()

    # update
    optimizer.step()
    total_loss+=loss.item()
  print(f"Epoch :{epoch+1}, loss: {total_loss/len(Train_Dataloader)}")

Epoch :1, loss: 1.828489847314411
Epoch :2, loss: 1.5932501779484276
Epoch :3, loss: 1.484299102084269
Epoch :4, loss: 1.396394208693306
Epoch :5, loss: 1.3277080925854825
Epoch :6, loss: 1.2927199597550902
Epoch :7, loss: 1.2543511908174858
Epoch :8, loss: 1.2152912862699
Epoch :9, loss: 1.1894697276583408
Epoch :10, loss: 1.175733830825076


In [ ]:
model.eval()

CIFAR_Model(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=2048, out_features=128, bias=True)
    (2): BatchNorm1d(128, eps

In [ ]:
total=0
correct=0
with torch.no_grad():

  for feature,label in Test_Dataloader:

    feature,label=feature.to(device),label.to(device)

    y_pred=model(feature)

    _,predict=torch.max(y_pred,1)

    total+=len(feature)

    correct+=(predict==label).sum().item()

  accuracy=correct/total
print(accuracy)

0.7117


In [ ]:
total=0
correct=0
with torch.no_grad():

  for feature,label in Train_Dataloader:

    feature,label=feature.to(device),label.to(device)

    y_pred=model(feature)

    _,predict=torch.max(y_pred,1)

    total+=len(feature)

    correct+=(predict==label).sum().item()

  accuracy=correct/total
print(accuracy)

0.76572
